![Week 1 Day 1](../assets/banners/w1d1.png)

# Week 1 Day 1 - Welcome, Setup and Your First Call to an LLM

Welcome to the very first lab! 🎉

This is where our six-week journey to build **GradeWise** begins. You've watched the demo: an agent that reads a stack of handwritten exam scripts, marks every answer against the examiner's rubric, backs each mark up with evidence from the course material, and writes a report card for every student. By week 6, you'll have built every piece of it yourself.

Today is about two things. First, making sure your machine is ready, so that nothing technical gets in the way for the rest of the course. Second, getting an early win: your first call to a large language model, and even a tiny first taste of AI grading.

![Your 6-week roadmap](../assets/roadmap/week1.png)

### 🎯 By the end of this lab you will
1. ✅ Confirm your environment is set up correctly (Python 3.12, the `.venv` kernel, your API key)
2. 🧠 Make your first call to an LLM with LangChain's `init_chat_model`
3. 🔍 Look inside a model's response: content, token usage and metadata
4. ⚖️ Ask a model to mark a student's answer, and spot why that's not yet good enough

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Run the cells <b>in order</b>, from top to bottom, with <b>Shift + Enter</b>. Before you start, check the top-right corner of this notebook: the kernel should say <code>.venv</code>. If it doesn't, click it → <b>Python Environments…</b> → pick the one in this project's <code>.venv</code> folder. Setup steps are in Part 6 of <code>setup/SETUP-PC.md</code> (or <code>-mac</code> / <code>-linux</code>).</span>
        </td>
    </tr>
</table>

## Part 1: Is this notebook using the course environment?

Every lab in this course runs inside the project's own private Python environment: the `.venv` folder that `uv sync` created. The cell below checks you're in the right one. It's the single most common setup problem, so let's rule it out first.

In [ ]:
import platform
import sys

print(f"Python {platform.python_version()} running from:\n  {sys.executable}\n")

if ".venv" in sys.executable and sys.version_info >= (3, 12):
    print("✅ Great: you're using the course's .venv kernel with Python 3.12+")
else:
    print("❌ This notebook is NOT using the course's .venv kernel.")
    print("   Fix: top-right of the notebook → Select Kernel → Python Environments → .venv")

## Part 2: Load your settings and API key

We keep secrets like API keys in a file called `.env`, never in the notebook itself. Two lines do the work:
- `load_dotenv(override=True)` reads `.env` into the environment. The `override=True` means that if you edit `.env`, re-running this cell picks up the change.
- Then we check the key exists, and print only its first few characters, never the whole thing.

Which key? The course runs on **OpenRouter**, one API in front of hundreds of models, so the default `MODEL_MODE`s (`cheap` and `free`, which you'll meet in a moment) both need just one key: `OPENROUTER_API_KEY`. Create it at [openrouter.ai/keys](https://openrouter.ai/keys) and paste it into `.env` (Part 5 of the setup guide walks you through it). `cheap` needs a few dollars of credits on the account; `free` needs none.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

from gradewise.config import settings

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"❌ {key_name} not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
        print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

Now let's meet **GradeWise's settings** (you just used them to find out which key to check). The course package `gradewise` (in the repo's `gradewise/` folder) reads your `.env` once and gives every lab the same configuration, including which models to use. Model names live *only* in `.env` and `gradewise/config.py`, so switching to another model, or calling OpenAI, Gemini, Claude or a free local model directly, never means editing a notebook.

In [ ]:
from gradewise import datasets
from gradewise.config import settings
from gradewise.console import success

success(
    "Your settings",
    {
        "Model mode": settings.model_mode,
        "Everyday model": settings.chat_model,
        "Grading model": settings.grading_model,
        "Vision model": settings.transcription_model,
        "Data size": settings.data_size,
    },
    next_step="Part 3: your first call to an LLM",
)
print(settings.describe_models())
print(datasets.describe())

### 💡 Fast & cheap run: two switches in `.env`

Two lines at the top of your `.env` decide what every lab uses, and what it costs:

| Line in `.env` | Options | What it does |
|---|---|---|
| `MODEL_MODE=cheap` | `cheap` (default) · `free` · `custom` | `cheap` uses OpenAI's `gpt-6-luna` through OpenRouter for everything: a call like today's costs well under a cent, a few cents for the whole week. `free` uses `openrouter/free`, which picks one of OpenRouter's free models for each request: $0, same key, no credits. `custom` uses whatever your `CHAT_MODEL`, `SMART_MODEL`, `VISION_MODEL` and `EMBEDDING_MODEL` lines say, e.g. OpenAI, Gemini or Claude directly, or Ollama. |
| `DATA_SIZE=full` | `full` (default) · `small` | `small` runs the labs on a slice of the data: 2 of the 4 course PDFs, questions Q1–Q3 and 2 of the 10 students. It's for when you know the ideas and just want to see the APIs work, fast and cheap. |

⚠️ **Free mode caveats:**
- **Rate limits:** 20 requests a minute and **50 a day** (1,000 a day once you've ever bought $10 of credits). GradeWise retries when it's told to slow down, and runs at most 2 calls at once.
- **Use it with `DATA_SIZE=small`:** a full class run makes hundreds of calls.
- **Quality and data:** the model can change from call to call, so results vary and an occasional structured-output failure is normal (GradeWise retries). Free providers may log or train on your prompts: fine for this course's made-up data, never for real student data. You may need to allow "free endpoints that may train on inputs" in your OpenRouter privacy settings.

To switch, edit `.env`, **save it**, then restart the kernel and run from the top. Day 2 looks at what each mode costs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Before running the next cell, predict: we're going to ask the model to explain what an operating system does <i>in one sentence, to a 12-year-old</i>. Will it obey the "one sentence" instruction? Will it use any jargon? Make your guess, then run it.</span>
        </td>
    </tr>
</table>

## Part 3: Your first call to an LLM

`init_chat_model` is LangChain's universal "give me a model" function. You pass a string in the form `provider:model`, like `openrouter:openai/gpt-6-luna` (provider `openrouter`, then OpenRouter's name for the model), and it returns a chat model with the same interface whatever the provider. Calling OpenAI directly instead (a `custom` option, with an `OPENAI_API_KEY`) is just `openai:gpt-6-luna`. That's why the whole course can swap providers in one line.

`.invoke(...)` sends a message and waits for the reply.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(settings.chat_model)

response = llm.invoke("In one sentence, explain to a 12-year-old what an operating system does.")
print(response.text)

🎉 **That's your first LLM call of the course!** Let's make it look nicer. Models often reply in Markdown, so we can render it:

In [ ]:
display(Markdown(response.text))

## Part 4: Look inside the response

`response` isn't just text: it's an `AIMessage` object, which carries useful information. The two things AI engineers look at constantly are **what the model said** and **how many tokens it cost**.

In [ ]:
print("Type of response:", type(response).__name__)
print("\nToken usage:", response.usage_metadata)
print("\nModel used:", response.response_metadata.get("model_name"))

💡 **`.text` or `.content`?** We read the reply with `response.text`. `response.content` is the raw payload: a plain string for many models, but a list of content blocks (text, reasoning, ...) for reasoning models and some providers, so `.text` is the safe way to get just the words.

💡 **Tokens** are the chunks of text that models read and write (roughly ¾ of a word each). You pay per token, for both input and output. We'll dig into tokens, context windows and cost properly on **Day 2**. For now, notice how *tiny* that call was: around a hundredth of a cent with `gpt-6-luna`, and nothing at all with `MODEL_MODE=free`.

### Messages with roles

Real applications send a list of messages with **roles**. The **system** message sets the model's character and rules; the **user** message is the request. We'll lean on system prompts heavily when we build GradeWise's examiner.

In [ ]:
messages = [
    {"role": "system", "content": "You are a friendly computer science tutor. Answer in at most two sentences."},
    {"role": "user", "content": "What's the difference between a process and a thread?"},
]

response = llm.invoke(messages)
display(Markdown(response.text))

## Part 5: A first taste of AI grading

Now for the fun part. Let's give the model what Emma, our CS201 teacher, would give a human marker: a **question**, a **marking scheme**, and a **student's answer**. Then we ask for a mark.

Here's a short answer from one of her students, Sam. (Typed for now: reading Sam's actual handwriting is week 4!)

In [ ]:
question = "Q3. What is a deadlock? Name TWO of the four necessary conditions for a deadlock. (4 marks)"

marking_scheme = """
- 2 marks: a correct definition (a set of processes, each waiting for a resource held by another, so none can proceed)
- 1 mark each (max 2): any two of mutual exclusion, hold and wait, no preemption, circular wait
"""

student_answer = """
A deadlock is when two programs are stuck because each one is waiting for the other one to finish.
Conditions: mutual exclusion and circular wait.
"""

grading_prompt = f"""You are an exam marker. Mark the student's answer using ONLY the marking scheme.

Question: {question}
Marking scheme: {marking_scheme}
Student answer: {student_answer}

Reply with the mark out of 4, then one sentence of feedback for the student."""

response = llm.invoke(grading_prompt)
display(Markdown(response.text))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Run that cell two or three more times. Does the mark always come out the same? Did the model give credit for "programs" instead of "processes"? Would you trust this to mark 200 scripts? Hold that thought: answering it properly is what the next six weeks are about.</span>
        </td>
    </tr>
</table>

### Why this isn't GradeWise (yet)

That worked surprisingly well for a few lines of code! But look at what's missing for a *real* grading system:

| Problem | Where we fix it |
|---|---|
| The reply is free text: hard for software to read the mark reliably | **Week 1, Days 3 & 5:** structured output with Pydantic |
| The model only knows what's in its training data, not *this* course's material | **Week 2:** RAG over the course PDFs |
| One prompt does everything: no checking, no retries, no human approval | **Week 3:** agents and LangGraph |
| Real answers are **handwritten** | **Week 4:** vision models |
| No evidence for the mark, and the mark may vary between runs | **Week 5:** evidence-based grading, consistency checks and evaluation against human marks |
| It's a notebook, not an app | **Week 6:** FastAPI + Streamlit, shipped in Docker |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Marking is one of the biggest time sinks in education: teachers routinely spend several hours a week on it. AI-assisted marking that is <b>transparent</b> (shows its evidence), <b>consistent</b> and keeps a <b>human in control</b> is a genuine product opportunity, and the same pattern (documents + rubric + evidence → structured judgement) applies to insurance claims, compliance reviews, code review and hiring screens.</span>
        </td>
    </tr>
</table>

## Part 6: The GradeWise way

Throughout the course, instead of calling `init_chat_model` directly, we'll get models from `gradewise.models`. It's the same thing underneath, but it reads your settings and gives a friendly, fixable error if a key is missing.

In [ ]:
from gradewise.models import chat_model

llm = chat_model()
print(llm.invoke("Reply with exactly: GradeWise is online ✅").text)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Open <code>gradewise/models.py</code> in the editor and have a look: it's short and heavily commented. This one file is why every lab can switch between OpenAI, Claude, Gemini and local models by editing a single line in <code>.env</code>.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

In this first lab you:
- ✅ confirmed your environment, settings and API key, and met the two `.env` switches, `MODEL_MODE` and `DATA_SIZE`
- 🧠 made your first LLM calls with `init_chat_model`, with and without roles
- 🔍 looked inside an `AIMessage`: content, token usage and metadata
- ⚖️ had a model mark Sam's deadlock answer, and listed everything a real grader still needs

**Tomorrow (Day 2):** how LLMs actually work under the hood (tokens, context windows, temperature and cost), and a tour of the LangChain 1.x ecosystem. Then we'll stream, batch and swap between providers.

⏭ **Next:** `week1/day2.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Make the marker <b>stricter</b>. Edit the grading prompt so the model only gives the definition marks if the student says <i>processes</i> (not "programs"), and must explain any mark it withholds. Run it a few times: is it more consistent? Bonus: write a second, weaker student answer and check the marks make sense. Solutions are in <code>week1/solutions/</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• LangChain models: <a href="https://docs.langchain.com/oss/python/langchain/models">docs.langchain.com/oss/python/langchain/models</a><br/>• Your OpenRouter key (one key for <code>cheap</code> and <code>free</code>): <a href="https://openrouter.ai/keys">openrouter.ai/keys</a><br/>• OpenRouter activity (keep an eye on spend!): <a href="https://openrouter.ai/activity">openrouter.ai/activity</a><br/>• Prefer another provider? <code>setup/SWAP_PROVIDER.md</code></span>
        </td>
    </tr>
</table>